### 1. Two Sum
Given an array of integers `nums` and an integer `target`, return _indices of the two numbers such that they add up to `target`_.

You may assume that each input would have **_exactly_ one solution**, and you may not use the _same_ element twice.

You can return the answer in any order.

**Example 1**:
```shell
Input: nums = [2,7,11,15], target = 9
Output: [0,1]
```
Explanation: Because `nums[0] + nums[1] == 9`, we return `[0, 1]`.

**Example 2**:
```shell
Input: nums = [3,2,4], target = 6
Output: [1,2]
```

**Example 3**:
```shell
Input: nums = [3,3], target = 6
Output: [0,1]
```

**Constraints:**

- `2 <= nums.length <= 104`
- `-109 <= nums[i] <= 109`
- `-109 <= target <= 109`
- **Only one valid answer exists.**

**Follow-up:** Can you come up with an algorithm that is less than $O(n2)$ time complexity?

---

### Reasoning
Notice this is not Two Sum II from last session — that one gave you a sorted array. This one doesn't say sorted.

**Step 1** — restate the problem in your own words, and ask me the clarifying questions an interviewer would expect (input bounds, duplicates, empty input, output order, etc.).
- So, the input is an array of whole numbers, are they positive only or negative and zeros possible as well? What are the bounds on input size? Can the input be empty of None? Can there be duplicates? As I understand the array is not sorted, correct? I need to return an array of integers specifying indices of those input array elements that add up to another input integer 'target'. So, two integer add up to target is what I need to find and return their indices in a list. I understand I can not use the same element twice as well? Does the output order matter?

Good clarifying questions — that's exactly what Step 1 is for. Answering as the interviewer would:

- **Values**: can be positive, negative, or zero.
- **Bounds**: array length up to `~10⁴` for this exercise.
- **Empty/None**: no, assume the array is valid with at least 2 elements.
- **Duplicates**: yes, the array can contain duplicate values.
- **Sorted**: correct, no guarantee — assume unsorted.
- **Same element twice**: correct, you cannot use the same index twice, even if the value could pair with itself (e.g., `target=6`, one element is 3 — you'd need a _different_ index also holding 3).
- **Number of solutions**: assume exactly one valid pair exists for this version — you don't need to handle "no answer found."
- **Output order**: doesn't matter which index comes first in your returned pair.

**Step 2** — work through one clean example by hand and state the expected output. Then layer on edge cases from what we just discussed (duplicates, negatives) on top of it.
- indices `0` and `1` and values `nums[0]` and `nums[1]` sum up to `target`, so the return value is a list of those indices `[0, 1]`. With duplicates in the input we actually can use them and not dropping them, say `[3, 3]` is a valid input for target `6`, that returns `[0, 1]`. Negatives also valid, e.g. `[-3, 0, 2]` `target = -1` returns `[0, 2]`.

Good — Step 2 is solid: `[0,1]` for the base case, `[3,3]` → `[0,1]` for duplicates, `[-3,0,2]` → `[0,2]` for negatives. And right, you caught it yourself — duplicate values aren't a problem here because you're comparing by _index_ pairs, not values, so `nums[0]==nums[1]==3` still gives two distinct, valid indices. That duplicate-skip logic you were thinking of belongs to 3Sum, where it prevents duplicate _output triplets_ — a different problem than this one has.

**Step 3** — brute force. Trace your nested-loop approach through `nums = [2,7,11,15]`, `target = 9`, listing out each `(i, j)` pair it actually checks and the sum computed. Then answer: why is this $O(n²)$, and what work is being repeated unnecessarily as `i` advances?
- So, with nested loops, we have the following pairs: `2-7` to `9`; `2-11` to `13`; `2-15` to `17`; `7-11` to `18`; `7-15` to `22`; `11-15` to `26`. This is $O(n^2)$ quadratic time since we test `n-1` items for each `n-1`. With this approach it would probably recalculate already computed pairs if there duplicates exist, it could be really redundant and unnecessary, e.g. `[1, 1, 1, 3, 3, 3]` will be scanned so the same pairs will be computed over and over.

Pairs and $O(n²)$ count are right. But look again at the "redundant work" question — duplicates aren't actually the source of inefficiency here. Even with zero duplicate values, this is still $O(n²)$ and still wasteful.

Look at what the inner loop is doing for each `i`: it's **checking, for every element already visited or not**, "does some element equal `target - nums[i]`?" What operation is that, in terms of what we just spent the whole last conversation talking about — array lookup by index, or array _search_ for a value?

And if that operation is the expensive one... what data structure did we just establish turns exactly that operation into $O(1)$?
- `j` is checking every element making it $O(n^2)$. This is the core of inefficiency of the quadratic time in this approach. Linear time for linear time. Array search for value. We could use a hash map data structure to store the current value mapping it to its index on each iteration and before it to compute `target - nums[i]` wondering 'have I seen current's value complement before' so it adds up to target, and if so return its index with current value index. They both point to valid elements. This current's value complement look up in dict will cost $O(1)$, as well as complement computation is also constant so does storing the current value in dict.

> The full optimized approach (and you derived it yourself): for each `i`, compute `complement = target - nums[i]`, check if `complement` is already a key in the dict ($O(1)$) — if yes, return the two indices; if no, store `nums[i] → i` and keep going. You traded the inner loop's $O(n)$ _search_ for an $O(1)$ _lookup_, one pass instead of two nested passes.

**Brute Force**
```py
def two_sum(nums: list[int], target: int) -> list[int]:
    # scan input with for loop - [0, len(nums) - 1] - to pre-last item
    # scan input with for loop - [i + 1, len(nums)] - from the next item after 'i'
    # test: if nums[i] == nums[j] return their indices
    # otherwise return None implicitly or anything else
```
[Visual](../images/1TwoSumBrute.png)

Before any code — **Step 4** requires you to state this out loud first:

1. What's the target **time** complexity now, and **space** complexity, for this hash map approach?
2. Compress the key insight into one sentence — the thing that makes this work.
- The trade-off is to use extra space for storing values with their indices in the dict to make this approach $O(n)$ time. Potentially if the valid pairs are at the end of the input array we would need to store every items with its index in the dict, that cost $O(n)$ space.

Time and space both $O(n)$ — confirmed, and your reasoning for space is right (worst case, you store almost every element before finding the match, e.g. if the pair is the last two elements).

Stated formally, per the required format:
```shell
Time:  O(n) — single pass, each element hashed and looked up once, O(1) per operation
Space: O(n) — worst case stores up to n-1 entries in the dict before finding the pair
```
> **Key Insight**: hash map look up instead of the inner loop scanning - one pass instead of of two nested passes.

Good, that's the key insight. Step 4 is done.

**Step 5** — define the function signature. Name, parameters with types, return type. What does it look like?
```py
def two_sum(nums: list[int], target: int) -> list[int]:
    pass
```
**Step 6** — before writing real code, outline it as pseudocode or comment skeleton. What are the steps, in order, inside this function?
```py
def two_sum(nums: list[int], target: int) -> list[int]:
    # set dict object: bucket = {}
    # scan input - for loop - full scan
    # compute complement for current element so they add up to target: target - nums[i]
    # test, if comlement IN bucket it means it is the correct pair - pick its index
    # otherwise, store to bucket the current element value mapping to its index
```

[Visual](../images/1TwoSumHash.png)

Correct outline, matches the $O(n)$ plan from Step 4. Order and logic check out.

**Step 7** — write the actual code.

Hash map - a complement

In [ ]:
def two_sum(nums: list[int], target: int) -> list[int]:
    seen = {}

    for i in range(len(nums)):
        compl = target - nums[i]

        if compl in seen:
            return [seen[compl], i]
        seen[nums[i]] = i

In [4]:
# two_sum([2,7,11,15], 9)
# two_sum([3,3], 6)
two_sum([-3,0,2], -1)

[0, 2]

**Optimization & Improvement**

Using `enumerate` function instead of `range` to directly get index and value.

In [ ]:
def func(nums: list[int], t: int) -> list[int] | None:
    seen = {}

    for i, value in enumerate(nums):
        compl = t - value

        if compl in seen:
            return [seen[compl], i]
        seen[value] = i

In [6]:
func([2,7,11,15], 9)
# func([3,3], 6)
# func([-3,0,2], -1)

[0, 1]

**Step 8** — trace this exact code through the three examples you already worked out in Step 2 (`[2,7,11,15]`/9, `[3,3]`/6, `[-3,0,2]`/-1). For each, walk through what `bucket` contains after each iteration and confirm the return value matches what you predicted earlier.

**Step 9** — final complexity confirmation against the actual code: does a single `for i in range(len(nums))` loop with dict `in` and dict assignment inside match your earlier claim of $O(n)$ time / $O(n)$ space? Briefly, why?
- Yes, a single for loop + `in` and assignment is exactly $O(n)$ time and space. Traversing once the whole array takes **n** operations, math operations and assignment statements are constant time, `compl` holds auxiliary space for math results, membership test `in` for dicts and sets is extremely fast since it uses hash value for look up as we discussed earlier about hash table's ops and hash function. Total space is $O(n)$ since we could store **n** items in bucket as we go along and look for complement for the current array's item.

**Complexity confirmed and correctly reasoned** — this closes out the problem.

> **Name the pattern:** this is the **one-pass hash map / complement lookup** pattern. The general shape: whenever a brute force needs to ask "have I seen X before?" or "does the complement of X exist somewhere in this array?", a hash map turns that $O(n)$ _search_ into an $O(1)$ _lookup_ (`in`), collapsing $O(n²)$ into $O(n)$. File it — you'll see this shape again anytime the brute force is nested loops built around "does some other element satisfy a condition with the current one."

#### Summary

**What you learned today:**
- the precise difference between **array position-access** ($O(1)$) and **array content-search** ($O(n)$);
- the actual mechanism behind hash map $O(1)$ — hashing, modulo into slots, collision resolution, load factor, resizing, and the worst-case $O(n)$ degeneration (hash flooding);
- and now applied all of that to solve **Two Sum** in one pass instead of two nested loops.